# Notebook 2 — Eksplorasi Data Mentah LIN/PBN

Notebook ini menampilkan contoh board dan struktur record hasil parsing LIN/PBN sebelum DDS, pembuatan label, dan pembagian dataset. Notebook ini tidak bergantung pada file `data/processed/` yang baru dibuat di Notebook 03.

**Input:** `data/raw/` (LIN) dan `data/raw_pbn/` (PBN).
**Tahap berikutnya:** Notebook 03 membentuk label DDS, dua varian dataset, dan split deal-aware.

---
## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

_root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _root_candidates if (p / "configs" / "config.yaml").exists() and (p / "src").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Tidak menemukan root repo; jalankan notebook dari repo atau folder notebooks.")
sys.path.insert(0, str(ROOT))

print(f"Repo root: {ROOT}")
print("Notebook ini hanya membaca sampel sumber LIN/PBN.")

---
## 1. Tabel Board Mentah Sebelum Feature Engineering

Satu baris = satu board hasil parsing. Tangan ditulis per suit (`♠ | ♥ | ♦ | ♣`), dan auction ditulis urut dari dealer. Sampel berasal dari beberapa file LIN dan PBN; dataset lengkap dibangun pada Notebook 03.

In [ ]:
from src.parser import LINParser, PBNParser

lin_sample_files = sorted((ROOT / 'data' / 'raw').glob('*.lin'))[:3]
pbn_sample_files = sorted({p for p in (ROOT / 'data' / 'raw_pbn').iterdir()
                           if p.suffix.lower() == '.pbn'})[:2]

raw_boards = []
for f in lin_sample_files:
    raw_boards += LINParser().parse_file(f)
for f in pbn_sample_files:
    raw_boards += PBNParser().parse_file(f)

def hand_str(hand):
    if hand is None:
        return ''
    d = hand.as_dict()
    return ' | '.join(''.join(d.get(s, [])) or '-' for s in ('S', 'H', 'D', 'C'))

def contract_str(b):
    if not b.contract_level:
        return 'PASS'
    return f"{b.contract_level}{b.contract_strain}{'xx' if b.redoubled else 'x' if b.doubled else ''}"

raw_df = pd.DataFrame([{
    'file'     : Path(b.source_file).name,
    'room'     : b.room,
    'board'    : b.board_number,
    'dealer'   : b.dealer,
    'vul'      : b.vulnerability,
    'North'    : hand_str(b.hands.get('N')),
    'East'     : hand_str(b.hands.get('E')),
    'South'    : hand_str(b.hands.get('S')),
    'West'     : hand_str(b.hands.get('W')),
    'lelang'   : ' '.join(b.auction),
    'kontrak'  : contract_str(b),
    'declarer' : b.declarer,
    'hasil'    : b.result,
    'trik'     : b.tricks_made,
} for b in raw_boards])

print(f'{len(raw_boards)} board dari {len(lin_sample_files)} file LIN + {len(pbn_sample_files)} file PBN')
display(raw_df.head(15))

In [ ]:
# Board yang sama dari sisi PBN
display(raw_df[raw_df['file'].str.lower().str.endswith('.pbn')].head(10))